# Discovery Feed Exploration

This notebook explores article embeddings, visualizes them with UMAP, and runs a few example queries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import umap

from src.data_loader import load_ag_news_sample
from src.embeddings import generate_embeddings
from src.vector_index import VectorIndex
from src.retrieval import retrieve_candidates
from src.ranking import rank_candidates

In [ ]:
articles = load_ag_news_sample()
texts = (articles['title'].fillna('') + '. ' + articles['description'].fillna('')).tolist()
articles.head()

In [ ]:
embeddings = generate_embeddings(texts)
embeddings.shape

In [ ]:
index = VectorIndex.build(embeddings)

In [ ]:
category_col = "topic" if "topic" in articles.columns else "category" if "category" in articles.columns else "label"
reducer = umap.UMAP(n_neighbors=10, min_dist=0.2, metric="cosine", random_state=42)
points = reducer.fit_transform(embeddings)

plt.figure(figsize=(8, 6))
for category in articles[category_col].unique():
    idx = articles[category_col] == category
    plt.scatter(points[idx, 0], points[idx, 1], s=30, alpha=0.8, label=category)
plt.title("Embedding Space by Category (UMAP Projection)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
query = "deep learning"
candidates = retrieve_candidates(query, index, articles, top_k=20)
ranked = rank_candidates(candidates, top_k=10)
ranked[["id", "title", category_col, "score"]].head(10)